# 8-Puzzle Solver using BFS, DFS and A*
## A Comparative Analysis of Search Strategies

### Step 1: Set up the Environment

Python is used to implement the 8-puzzle solver. Google Colab is used as the programming environment.

The 8-puzzle board is represented as a 3×3 grid containing 8 numbered tiles and one blank space. In the program, the blank space is represented by `0`.

The board is stored as a tuple containing 9 elements so that each puzzle configuration can be represented as a single state.

In [ ]:
from collections import deque
import heapq
import time

## Step 2: Formulate the State Space

A state represents one particular configuration of the 8-puzzle.

For example:

    1 2 3
    4 0 6
    7 5 8

can be represented as:

`(1, 2, 3, 4, 0, 6, 7, 5, 8)`

Here, `0` represents the blank space.

The initial state is the configuration from which the search begins.

The goal state is the standard solved configuration:

    1 2 3
    4 5 6
    7 8 0

The valid actions are moving the blank space Up, Down, Left, or Right, provided that the movement stays within the 3×3 board.

The successor function generates the new states that can be reached from the current state by making one valid move.

A goal test checks whether the current state is equal to the goal state.

In [ ]:
# Goal state
GOAL = (
    1, 2, 3,
    4, 5, 6,
    7, 8, 0
)

# Initial state
initial_state = (
    1, 2, 3,
    4, 0, 6,
    7, 5, 8
)

### Displaying the Puzzle Board

Although the puzzle is stored internally as a tuple of 9 elements, it is easier to understand when displayed as a 3×3 grid.

A function is therefore created to display the state in its original 3×3 form.

In [ ]:
def print_board(state):
    for i in range(0, 9, 3):
        print(state[i:i+3])
    print()

In [ ]:
print("Initial State:")
print_board(initial_state)

print("Goal State:")
print_board(GOAL)

Initial State:
(1, 2, 3)
(4, 0, 6)
(7, 5, 8)

Goal State:
(1, 2, 3)
(4, 5, 6)
(7, 8, 0)



### Goal Test

The goal test determines whether the current puzzle configuration has reached the required goal configuration.

The current state is compared with the predefined goal state.

If both states are equal, the puzzle is solved; otherwise, the search continues.

In [ ]:
def is_goal(state):
    return state == GOAL

In [ ]:
print("Is initial state the goal?", is_goal(initial_state))
print("Is goal state the goal?", is_goal(GOAL))

Is initial state the goal? False
Is goal state the goal? True


### Successor Function

The successor function generates all valid states that can be reached from the current state in one move.

The blank space can move in four possible directions:

- Up
- Down
- Left
- Right

However, a movement is valid only when the blank remains inside the 3×3 board.

For every valid movement, the blank space is swapped with the corresponding tile. The resulting configuration is a new successor state.

The successor function is common to BFS, DFS and A*. The three algorithms differ only in the way they select which state to explore next.

In [ ]:
def get_neighbors(state):

    neighbors = []

    # Find the position of the blank
    blank = state.index(0)

    # Convert the position into row and column
    row = blank // 3
    col = blank % 3

    # Possible moves of the blank
    moves = [
        ("Up", -1, 0),
        ("Down", 1, 0),
        ("Left", 0, -1),
        ("Right", 0, 1)
    ]

    for move, row_change, col_change in moves:

        new_row = row + row_change
        new_col = col + col_change

        # Check if the new position is inside the board
        if 0 <= new_row < 3 and 0 <= new_col < 3:

            # Convert row and column back to index
            new_blank = new_row * 3 + new_col

            # Make a copy of the current state
            new_state = list(state)

            # Swap blank with the tile
            new_state[blank], new_state[new_blank] = \
                new_state[new_blank], new_state[blank]

            # Store the new state and the move
            neighbors.append(
                (tuple(new_state), move)
            )

    return neighbors

In [ ]:
neighbors = get_neighbors(initial_state)

print("Initial State:")
print_board(initial_state)

print("Possible Successor States:")

for state, move in neighbors:

    print("Move:", move)
    print_board(state)

Initial State:
(1, 2, 3)
(4, 0, 6)
(7, 5, 8)

Possible Successor States:
Move: Up
(1, 0, 3)
(4, 2, 6)
(7, 5, 8)

Move: Down
(1, 2, 3)
(4, 5, 6)
(7, 0, 8)

Move: Left
(1, 2, 3)
(0, 4, 6)
(7, 5, 8)

Move: Right
(1, 2, 3)
(4, 6, 0)
(7, 5, 8)



## Step 3: Implement Breadth-First Search (BFS)

Breadth-First Search (BFS) is an uninformed search algorithm that explores the state space level by level.

BFS starts from the initial state and first explores all states that are one move away. It then explores all states that are two moves away, followed by states at greater depths.

BFS uses a FIFO (First-In, First-Out) queue. The state that is inserted into the queue first is explored first.

A visited set is maintained to prevent the same state from being explored repeatedly.

For the 8-puzzle, BFS generates valid successor states from the current state and adds unvisited states to the queue. The search continues until the goal state is reached.

Since every move in the 8-puzzle has the same cost, BFS can find a shortest solution path.

### Working of BFS

The BFS algorithm follows these steps:

1. Insert the initial state into a FIFO queue.
2. Mark the initial state as visited.
3. Remove the state at the front of the queue.
4. Check whether the removed state is the goal state.
5. If it is not the goal state, generate all valid successor states.
6. Add unvisited successor states to the queue.
7. Mark newly added states as visited.
8. Repeat the process until the goal state is found or the queue becomes empty.

The solution path is stored along with each state so that the sequence of moves leading from the initial state to the goal can be returned.

In [ ]:
def bfs(start):

    # Create a FIFO queue
    # Each item contains:
    # (current state, path used to reach the state)
    queue = deque()

    queue.append((start, []))

    # Set of states that have already been visited
    visited = {start}

    # Count the number of expanded nodes
    nodes_expanded = 0

    # Track maximum number of states in the queue
    max_frontier = 1

    # Continue while the queue is not empty
    while queue:

        # Remove the first item from the queue
        state, path = queue.popleft()

        # Count this state as expanded
        nodes_expanded += 1

        # Check whether the goal has been reached
        if is_goal(state):
            return path, nodes_expanded, max_frontier

        # Generate all valid successor states
        for next_state, move in get_neighbors(state):

            # Process only states that have not been visited
            if next_state not in visited:

                # Mark the state as visited
                visited.add(next_state)

                # Add the new move to the existing path
                new_path = path + [move]

                # Add the successor to the queue
                queue.append(
                    (next_state, new_path)
                )

        # Update maximum queue size
        max_frontier = max(
            max_frontier,
            len(queue)
        )

    # Return if no solution is found
    return None, nodes_expanded, max_frontier

### Important Components of BFS

**Queue:**

A FIFO queue is used so that states are explored in the same order in which they are discovered.

**Visited Set:**

The visited set stores states that have already been discovered. This prevents repeated exploration of the same puzzle configuration.

**Path:**

The path stores the sequence of moves used to reach the current state.

**Nodes Expanded:**

The number of states removed from the queue and examined by the algorithm is counted as the number of nodes expanded.

**Maximum Frontier Size:**

The largest number of states present in the queue at any point is recorded as the maximum frontier size.

**Goal Test:**

When a state is removed from the queue, it is checked against the goal state. If it matches the goal, the stored path is returned.

In [ ]:
# Run BFS on the initial state

bfs_path, bfs_nodes, bfs_frontier = bfs(initial_state)

print("BFS RESULT")
print("=" * 40)

print("Initial State:")
print_board(initial_state)

print("Goal State:")
print_board(GOAL)

print("Solution Path:")
print(bfs_path)

print("Solution Path Length:")
print(len(bfs_path))

print("Nodes Expanded:")
print(bfs_nodes)

print("Maximum Frontier Size:")
print(bfs_frontier)

BFS RESULT
Initial State:
(1, 2, 3)
(4, 0, 6)
(7, 5, 8)

Goal State:
(1, 2, 3)
(4, 5, 6)
(7, 8, 0)

Solution Path:
['Down', 'Right']
Solution Path Length:
2
Nodes Expanded:
9
Maximum Frontier Size:
8


### BFS Result

The BFS implementation returns the sequence of moves required to reach the goal state.

The following performance measures are also recorded:

- Solution path length
- Number of nodes expanded
- Maximum frontier size

These measurements will later be used to compare BFS with DFS and A*.

## Step 4: Implement Depth-First Search (DFS)

Depth-First Search (DFS) is an uninformed search algorithm that explores one branch of the state space as deeply as possible before backtracking.

DFS uses a LIFO (Last-In, First-Out) stack. The most recently added state is explored first.

For the 8-puzzle, DFS starts from the initial state and generates successor states. It continues exploring one branch until it reaches the goal state or reaches a specified depth limit.

A depth limit is used to prevent DFS from exploring indefinitely in the state space. For this implementation, a depth limit of 30 is used.

DFS maintains a visited set to avoid repeatedly exploring the same state.

Unlike BFS, DFS does not guarantee the shortest solution path.

### Working of DFS

The DFS algorithm follows these steps:

1. Insert the initial state into a LIFO stack.
2. Mark the initial state as visited.
3. Remove the most recently added state from the stack.
4. Check whether the state is the goal state.
5. If it is not the goal state, generate its valid successor states.
6. Add unvisited successor states to the stack.
7. Continue exploring the newest state first.
8. Stop when the goal state is found or the depth limit is reached.

The solution path is stored with each state so that the sequence of moves from the initial state to the goal can be returned.

In [ ]:
def dfs(start, depth_limit=30):

    # Stack stores: (state, path, depth)
    stack = [(start, [], 0)]

    # Store the minimum depth at which each state was visited
    visited_depth = {start: 0}

    nodes_expanded = 0
    max_frontier = 1

    while stack:

        state, path, depth = stack.pop()

        nodes_expanded += 1

        # Goal test
        if is_goal(state):
            return path, nodes_expanded, max_frontier

        # Depth limit
        if depth >= depth_limit:
            continue

        # Generate successors
        neighbors = get_neighbors(state)

        # Add successors to stack
        for next_state, move in reversed(neighbors):

            new_depth = depth + 1

            # Visit the state if:
            # 1. It has never been visited
            # OR
            # 2. We reached it at a smaller depth
            if (next_state not in visited_depth or
                    new_depth < visited_depth[next_state]):

                visited_depth[next_state] = new_depth

                new_path = path + [move]

                stack.append(
                    (next_state, new_path, new_depth)
                )

        # Record maximum frontier size
        max_frontier = max(max_frontier, len(stack))

    return None, nodes_expanded, max_frontier

### Important Components of DFS

**Stack:**

A LIFO stack is used so that the most recently generated state is explored first.

**Visited Set:**

The visited set stores states that have already been discovered and prevents repeated exploration.

**Depth Limit:**

A depth limit is used to prevent DFS from exploring indefinitely deep paths.

**Path:**

The path stores the sequence of moves used to reach the current state.

**Nodes Expanded:**

The number of states removed from the stack and examined by DFS.

**Maximum Frontier Size:**

The largest number of states present in the stack at any point.

**Goal Test:**

When a state is removed from the stack, it is checked against the goal state.

In [ ]:
dfs_path, dfs_nodes, dfs_frontier = dfs(initial_state)

print("DFS RESULT")
print("=" * 40)

print("Solution Path:")
print(dfs_path)

print("Solution Path Length:")
print(len(dfs_path))

print("Nodes Expanded:")
print(dfs_nodes)

print("Maximum Frontier Size:")
print(dfs_frontier)

DFS RESULT
Solution Path:
['Up', 'Left', 'Down', 'Down', 'Right', 'Up', 'Up', 'Right', 'Down', 'Left', 'Down', 'Left', 'Up', 'Right', 'Right', 'Up', 'Left', 'Left', 'Down', 'Right', 'Down', 'Right', 'Up', 'Left', 'Up', 'Left', 'Down', 'Right', 'Right', 'Down']
Solution Path Length:
30
Nodes Expanded:
146500
Maximum Frontier Size:
28


## Step 5: Implement A* Search

A* Search is an informed search algorithm that uses both the cost of reaching a state and an estimate of the remaining cost to the goal.

A* uses the evaluation function:

`f(n) = g(n) + h(n)`

where:

- `g(n)` is the actual cost from the initial state to the current state.
- `h(n)` is the estimated cost from the current state to the goal.
- `f(n)` is the total estimated cost of the solution through state `n`.

For the 8-puzzle, Manhattan distance is used as the heuristic.

The Manhattan distance of a tile is the number of horizontal and vertical moves required to place that tile in its goal position.

The blank tile (0) is not included in the heuristic calculation.

A priority queue is used to select the state with the smallest `f(n)` value.

### Manhattan Distance Heuristic

For each numbered tile, Manhattan distance is calculated as:

`|current_row - goal_row| + |current_column - goal_column|`

The heuristic value is obtained by adding the Manhattan distances of all numbered tiles.

Therefore:

`h(n) = Σ (|current_row - goal_row| + |current_column - goal_column|)`

The blank tile is ignored.

Manhattan distance is an admissible heuristic for the 8-puzzle because it never overestimates the minimum number of moves required to reach the goal.

In [ ]:
def manhattan_distance(state):

    distance = 0

    for tile in range(1, 9):

        # Current position of the tile
        current_index = state.index(tile)
        current_row = current_index // 3
        current_col = current_index % 3

        # Goal position of the tile
        goal_index = GOAL.index(tile)
        goal_row = goal_index // 3
        goal_col = goal_index % 3

        # Manhattan distance for this tile
        distance += abs(current_row - goal_row)
        distance += abs(current_col - goal_col)

    return distance

In [ ]:
print("Initial State:")
print_board(initial_state)

print("Manhattan Distance:")
print(manhattan_distance(initial_state))

print("Goal State:")
print_board(GOAL)

print("Manhattan Distance of Goal:")
print(manhattan_distance(GOAL))

Initial State:
(1, 2, 3)
(4, 0, 6)
(7, 5, 8)

Manhattan Distance:
2
Goal State:
(1, 2, 3)
(4, 5, 6)
(7, 8, 0)

Manhattan Distance of Goal:
0


### Working of A* Search

The A* algorithm follows these steps:

1. Insert the initial state into a priority queue.
2. Calculate its heuristic value `h(n)`.
3. Calculate `f(n) = g(n) + h(n)`.
4. Remove the state having the smallest `f(n)` value.
5. Check whether it is the goal state.
6. If it is not the goal state, generate its valid successor states.
7. Calculate the cost and heuristic value for each successor.
8. Insert the successors into the priority queue.
9. Repeat until the goal state is reached or the priority queue becomes empty.

A* uses the path cost and heuristic together to select promising states.

In [ ]:
def a_star(start):

    # Priority queue
    # Each item contains:
    # (f, g, state, path)
    priority_queue = []

    # Initial cost
    g = 0

    # Initial heuristic
    h = manhattan_distance(start)

    # f(n) = g(n) + h(n)
    f = g + h

    heapq.heappush(
        priority_queue,
        (f, g, start, [])
    )

    # Store the best known cost for each state
    cost_so_far = {start: 0}

    # Count expanded nodes
    nodes_expanded = 0

    # Track maximum frontier size
    max_frontier = 1

    while priority_queue:

        # Remove state with smallest f value
        f, g, state, path = heapq.heappop(priority_queue)

        # Count expanded node
        nodes_expanded += 1

        # Check goal
        if is_goal(state):
            return path, nodes_expanded, max_frontier

        # Generate successors
        for next_state, move in get_neighbors(state):

            # Cost of reaching successor
            new_g = g + 1

            # If this state is new or a better path is found
            if (next_state not in cost_so_far or
                    new_g < cost_so_far[next_state]):

                cost_so_far[next_state] = new_g

                # Calculate heuristic
                h = manhattan_distance(next_state)

                # Calculate f(n)
                new_f = new_g + h

                # Store updated path
                new_path = path + [move]

                heapq.heappush(
                    priority_queue,
                    (new_f, new_g, next_state, new_path)
                )

        # Update maximum frontier size
        max_frontier = max(
            max_frontier,
            len(priority_queue)
        )

    return None, nodes_expanded, max_frontier

In [ ]:
# Run A* on the initial state

astar_path, astar_nodes, astar_frontier = a_star(initial_state)

print("A* RESULT")
print("=" * 40)

print("Initial State:")
print_board(initial_state)

print("Goal State:")
print_board(GOAL)

print("Solution Path:")
print(astar_path)

print("Solution Path Length:")
print(len(astar_path))

print("Nodes Expanded:")
print(astar_nodes)

print("Maximum Frontier Size:")
print(astar_frontier)

A* RESULT
Initial State:
(1, 2, 3)
(4, 0, 6)
(7, 5, 8)

Goal State:
(1, 2, 3)
(4, 5, 6)
(7, 8, 0)

Solution Path:
['Down', 'Right']
Solution Path Length:
2
Nodes Expanded:
3
Maximum Frontier Size:
5


## Step 6: Prepare Test Cases

To compare BFS, DFS and A*, three different initial states are used.

The test cases represent different difficulty levels:

- Easy: approximately 4–6 moves from the goal
- Medium: approximately 8–12 moves from the goal
- Hard: approximately 15 or more moves from the goal

Each initial state will be solved using BFS, DFS and A*.

Therefore, a total of 9 experiments will be performed.

In [ ]:
# Test cases for the experiment

easy = (
    1, 5, 2,
    4, 0, 3,
    7, 8, 6
)

medium = (
    5, 8, 2,
    1, 0, 3,
    4, 7, 6
)

hard = (
    8, 7, 0,
    5, 4, 2,
    1, 6, 3
)

test_cases = {
    "Easy": easy,
    "Medium": medium,
    "Hard": hard
}

## Step 7: Run the Experiments

Each of the three search algorithms is executed on the Easy, Medium and Hard test cases.

For every run, the following performance measures are recorded:

- Number of nodes expanded
- Maximum frontier size
- Execution time
- Solution path length

These measurements will be used to compare the performance of BFS, DFS and A*.

In [ ]:
def run_experiment():

    results = []

    algorithms = {
        "BFS": bfs,
        "DFS": dfs,
        "A*": a_star
    }

    for difficulty, state in test_cases.items():

        for algorithm_name, algorithm in algorithms.items():

            start_time = time.perf_counter()

            path, nodes, frontier = algorithm(state)

            end_time = time.perf_counter()

            execution_time = end_time - start_time

            results.append({
                "Difficulty": difficulty,
                "Algorithm": algorithm_name,
                "Nodes Expanded": nodes,
                "Maximum Frontier": frontier,
                "Execution Time": execution_time,
                "Path Length": len(path) if path is not None else None
            })

    return results

In [ ]:
results = run_experiment()

for result in results:
    print(result)

{'Difficulty': 'Easy', 'Algorithm': 'BFS', 'Nodes Expanded': 24, 'Maximum Frontier': 18, 'Execution Time': 0.00010806099999172147, 'Path Length': 4}
{'Difficulty': 'Easy', 'Algorithm': 'DFS', 'Nodes Expanded': 49897, 'Maximum Frontier': 28, 'Execution Time': 0.11932672500006447, 'Path Length': 30}
{'Difficulty': 'Easy', 'Algorithm': 'A*', 'Nodes Expanded': 5, 'Maximum Frontier': 6, 'Execution Time': 8.761000026424881e-05, 'Path Length': 4}
{'Difficulty': 'Medium', 'Algorithm': 'BFS', 'Nodes Expanded': 540, 'Maximum Frontier': 378, 'Execution Time': 0.00165156000002753, 'Path Length': 10}
{'Difficulty': 'Medium', 'Algorithm': 'DFS', 'Nodes Expanded': 15510, 'Maximum Frontier': 28, 'Execution Time': 0.033064603000184434, 'Path Length': 30}
{'Difficulty': 'Medium', 'Algorithm': 'A*', 'Nodes Expanded': 14, 'Maximum Frontier': 12, 'Execution Time': 0.00018509799974708585, 'Path Length': 10}
{'Difficulty': 'Hard', 'Algorithm': 'BFS', 'Nodes Expanded': 37810, 'Maximum Frontier': 16993, 'Execu

In [ ]:
import pandas as pd

results_df = pd.DataFrame(results)

results_df

,Difficulty,Algorithm,Nodes Expanded,Maximum Frontier,Execution Time,Path Length
0,Easy,BFS,24,18,0.000172,4
1,Easy,DFS,58,76,0.000238,30
2,Easy,A*,5,6,0.000106,4
3,Medium,BFS,540,378,0.002181,10
4,Medium,DFS,10296,76,0.012596,30
5,Medium,A*,14,12,0.000159,10
6,Hard,BFS,37810,16993,0.218113,20
7,Hard,DFS,235340707,61,281.246512,30
8,Hard,A*,180,112,0.001914,20


## Step 9: Analysis of Results

The results obtained from BFS, DFS and A* are compared using the recorded performance measures.

The comparison considers:

1. Which algorithm expands the fewest nodes.
2. Which algorithm requires the least frontier memory.
3. Which algorithm has the lowest execution time.
4. Whether the algorithms produce a shortest solution path.
5. How the performance changes as the puzzle difficulty increases.

BFS is complete and, because every move has equal cost, it can find an optimal shortest path. However, its frontier can become large as the search depth increases.

DFS generally requires less frontier memory than BFS, but it does not guarantee an optimal solution path. Its performance can also depend strongly on the order in which successor states are explored.

A* uses heuristic information to guide the search toward the goal. With the Manhattan distance heuristic, A* can efficiently find an optimal solution for the 8-puzzle while usually exploring fewer unnecessary states than uninformed search.

## Step 10: Conclusion

The 8-puzzle was represented as a state-space search problem and solved using BFS, DFS and A* search strategies.

BFS explores the state space level by level using a FIFO queue. DFS explores deeply using a LIFO stack and a depth limit. A* uses heuristic information to guide the search using the evaluation function:

`f(n) = g(n) + h(n)`

where `g(n)` represents the actual path cost and `h(n)` represents the estimated remaining cost.

For A*, Manhattan distance was used as the heuristic:

`h(n) = Σ (|current_row - goal_row| + |current_column - goal_column|)`

The performance of the three algorithms was compared using nodes expanded, maximum frontier size, execution time and solution path length.

Execution time was calculated as:

`Execution Time = End Time - Start Time`

Solution path length was calculated as the number of moves from the initial state to the goal state.

Based on the experimental results, the search strategies are compared in terms of efficiency, memory usage, execution time and solution quality.

A* is generally well suited for the 8-puzzle because it uses heuristic information to guide the search. BFS can be preferred when finding the shortest solution without using heuristic information is required, while DFS can be useful when memory usage is an important consideration.